# MedGemma Hyperparameter Analysis

**Interactive notebook for analyzing hyperparameter tuning experiments**

This notebook provides comprehensive tools for:
- Loading and exploring tuning results
- Interactive visualizations (Plotly)
- Parameter importance analysis
- High-dimensional exploration (HiPlot)
- Experiment comparison
- Report generation

---

## 1. Setup and Imports

In [ ]:
# Standard library imports
import os
import sys
import json
from pathlib import Path
from datetime import datetime

# Data handling
import numpy as np
import pandas as pd

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

# HiPlot for high-dimensional visualization
try:
    import hiplot as hip
    HIPLOT_AVAILABLE = True
except ImportError:
    HIPLOT_AVAILABLE = False
    print("HiPlot not available. Install with: pip install hiplot")

# MedGemma optimization modules
sys.path.insert(0, str(Path.cwd().parent))
from medai_compass.optimization import (
    HPAnalyzer,
    SearchSpaceConfig,
    TuneConfig,
    HyperparameterTuner,
    create_asha_scheduler,
    create_optuna_search,
    create_pbt_scheduler,
)

# Set Plotly to render in notebook
import plotly.io as pio
pio.renderers.default = "notebook"

print("✅ All imports successful!")

## 2. Configuration

Configure paths and settings for your tuning experiments.

In [ ]:
# Configuration
CONFIG = {
    # Path to Ray Tune results directory
    "results_path": "../ray_results",
    
    # Model selection
    "model_name": "medgemma-4b",  # or "medgemma-27b"
    
    # Primary metric for optimization
    "primary_metric": "eval_loss",
    "metric_mode": "min",  # "min" or "max"
    
    # Output directory for saved figures
    "output_dir": "../outputs/hp_analysis",
}

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)

print(f"Model: {CONFIG['model_name']}")
print(f"Primary metric: {CONFIG['primary_metric']} ({CONFIG['metric_mode']})")
print(f"Results path: {CONFIG['results_path']}")

### Interactive Configuration Widget

In [ ]:
# Interactive configuration widget
model_dropdown = widgets.Dropdown(
    options=["medgemma-4b", "medgemma-27b"],
    value=CONFIG["model_name"],
    description="Model:",
    style={"description_width": "initial"},
)

metric_dropdown = widgets.Dropdown(
    options=["eval_loss", "train_loss", "accuracy", "f1_score"],
    value=CONFIG["primary_metric"],
    description="Metric:",
    style={"description_width": "initial"},
)

mode_toggle = widgets.ToggleButtons(
    options=["min", "max"],
    value=CONFIG["metric_mode"],
    description="Mode:",
    style={"description_width": "initial"},
)

results_path_text = widgets.Text(
    value=CONFIG["results_path"],
    description="Results Path:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="50%"),
)

def update_config(change):
    CONFIG["model_name"] = model_dropdown.value
    CONFIG["primary_metric"] = metric_dropdown.value
    CONFIG["metric_mode"] = mode_toggle.value
    CONFIG["results_path"] = results_path_text.value

model_dropdown.observe(update_config, names="value")
metric_dropdown.observe(update_config, names="value")
mode_toggle.observe(update_config, names="value")
results_path_text.observe(update_config, names="value")

config_box = widgets.VBox([
    widgets.HTML("<h4>Configuration</h4>"),
    widgets.HBox([model_dropdown, metric_dropdown]),
    mode_toggle,
    results_path_text,
])

display(config_box)

## 3. Load Experiment Results

Load results from completed hyperparameter tuning experiments.

In [ ]:
# Initialize analyzer
analyzer = HPAnalyzer()

# Check if results path exists
results_path = Path(CONFIG["results_path"])

if results_path.exists():
    # Load from Ray Tune results
    try:
        analyzer.load_results_from_path(str(results_path))
        print(f"✅ Loaded {analyzer.num_trials} trials from {results_path}")
    except Exception as e:
        print(f"⚠️ Could not load from path: {e}")
        print("Creating sample data for demonstration...")
        # Create sample data for demonstration
        sample_results = generate_sample_results()
        analyzer.load_results_from_dict(sample_results)
else:
    print(f"⚠️ Results path not found: {results_path}")
    print("Creating sample data for demonstration...")
    
    # Generate sample data for demonstration
    def generate_sample_results(n_trials=50):
        """Generate sample tuning results for demonstration."""
        np.random.seed(42)
        results = []
        
        for i in range(n_trials):
            # Sample hyperparameters
            lr = np.random.choice([1e-5, 5e-5, 1e-4, 2e-4, 5e-4, 1e-3])
            lora_r = np.random.choice([8, 16, 32, 64])
            lora_alpha = np.random.choice([16, 32, 64, 128])
            weight_decay = np.random.choice([0.0, 0.01, 0.05, 0.1])
            batch_size = np.random.choice([2, 4, 8])
            lora_dropout = np.random.choice([0.0, 0.05, 0.1])
            
            # Simulate metrics (lower loss for optimal configs)
            # Optimal: lr around 1e-4, lora_r around 32
            base_loss = 0.5
            lr_factor = abs(np.log10(lr) + 4) * 0.1
            rank_factor = abs(lora_r - 32) * 0.005
            noise = np.random.normal(0, 0.05)
            eval_loss = base_loss + lr_factor + rank_factor + noise
            train_loss = eval_loss - 0.05 + np.random.normal(0, 0.02)
            
            results.append({
                "trial_id": f"trial_{i:03d}",
                "config": {
                    "learning_rate": lr,
                    "lora_r": lora_r,
                    "lora_alpha": lora_alpha,
                    "weight_decay": weight_decay,
                    "batch_size": batch_size,
                    "lora_dropout": lora_dropout,
                },
                "metrics": {
                    "eval_loss": max(0.1, eval_loss),
                    "train_loss": max(0.05, train_loss),
                },
                "status": "COMPLETED",
            })
        
        return results
    
    sample_results = generate_sample_results()
    analyzer.load_results_from_dict(sample_results)
    print(f"✅ Created {analyzer.num_trials} sample trials for demonstration")

In [ ]:
# Display overview
print("\n" + "="*60)
print("EXPERIMENT OVERVIEW")
print("="*60)
print(f"Total trials: {analyzer.num_trials}")
print(f"Parameters: {', '.join(analyzer.param_columns)}")
print(f"Metrics: {', '.join(analyzer.metric_columns)}")

# Display dataframe preview
print("\n" + "-"*60)
print("DATA PREVIEW")
print("-"*60)
display(analyzer.dataframe.head(10))

## 4. Best Configuration

In [ ]:
# Get best trial
best_trial = analyzer.get_best_trial(
    metric=CONFIG["primary_metric"],
    mode=CONFIG["metric_mode"],
)

print("\n" + "="*60)
print("🏆 BEST CONFIGURATION")
print("="*60)
print(f"Trial ID: {best_trial.trial_id}")
print(f"\n📊 Metrics:")
for metric, value in best_trial.metrics.items():
    print(f"  • {metric}: {value:.6f}")

print(f"\n⚙️ Configuration:")
for param, value in best_trial.config.items():
    if isinstance(value, float):
        print(f"  • {param}: {value:.2e}")
    else:
        print(f"  • {param}: {value}")

## 5. Interactive Visualizations

Explore hyperparameter tuning results with interactive Plotly charts.

### 5.1 Optimization History

In [ ]:
# Optimization history plot
fig_history = analyzer.plot_optimization_history(
    metric=CONFIG["primary_metric"],
    mode=CONFIG["metric_mode"],
)
fig_history.show()

### 5.2 Parallel Coordinates Plot

In [ ]:
# Interactive parameter selection for parallel coordinates
param_checkboxes = widgets.SelectMultiple(
    options=analyzer.param_columns,
    value=analyzer.param_columns[:6],  # Select first 6 by default
    description="Parameters:",
    style={"description_width": "initial"},
    layout=widgets.Layout(height="150px"),
)

output_parcoords = widgets.Output()

def update_parcoords(change):
    with output_parcoords:
        output_parcoords.clear_output(wait=True)
        fig = analyzer.plot_parallel_coordinates(
            color_metric=CONFIG["primary_metric"],
            params=list(param_checkboxes.value),
        )
        fig.show()

param_checkboxes.observe(update_parcoords, names="value")

# Initial plot
with output_parcoords:
    fig = analyzer.plot_parallel_coordinates(
        color_metric=CONFIG["primary_metric"],
    )
    fig.show()

display(widgets.VBox([param_checkboxes, output_parcoords]))

### 5.3 Parameter Importance

In [ ]:
# Calculate and display parameter importance
importance = analyzer.calculate_parameter_importance(
    metric=CONFIG["primary_metric"]
)

print("\n" + "="*60)
print("PARAMETER IMPORTANCE")
print("="*60)

# Sort by importance
sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
for param, score in sorted_importance:
    bar = "█" * int(score * 20)
    print(f"{param:20s} {bar:20s} {score:.4f}")

# Plot importance
fig_importance = analyzer.plot_parameter_importance(
    metric=CONFIG["primary_metric"]
)
fig_importance.show()

### 5.4 Interactive Contour Exploration

In [ ]:
# Interactive contour plot widget
param1_dropdown = widgets.Dropdown(
    options=analyzer.param_columns,
    value=analyzer.param_columns[0] if analyzer.param_columns else None,
    description="Parameter 1:",
    style={"description_width": "initial"},
)

param2_dropdown = widgets.Dropdown(
    options=analyzer.param_columns,
    value=analyzer.param_columns[1] if len(analyzer.param_columns) > 1 else None,
    description="Parameter 2:",
    style={"description_width": "initial"},
)

contour_output = widgets.Output()

def update_contour(change):
    with contour_output:
        contour_output.clear_output(wait=True)
        try:
            fig = analyzer.plot_contour(
                param1=param1_dropdown.value,
                param2=param2_dropdown.value,
                metric=CONFIG["primary_metric"],
            )
            fig.show()
        except Exception as e:
            print(f"Error: {e}")

param1_dropdown.observe(update_contour, names="value")
param2_dropdown.observe(update_contour, names="value")

# Initial plot
update_contour(None)

display(widgets.VBox([
    widgets.HTML("<h4>Select parameters to compare:</h4>"),
    widgets.HBox([param1_dropdown, param2_dropdown]),
    contour_output,
]))

## 6. High-Dimensional Visualization (HiPlot)

Use HiPlot for interactive high-dimensional parameter exploration.

In [ ]:
if HIPLOT_AVAILABLE:
    # Create HiPlot experiment
    exp = analyzer.create_hiplot_experiment()
    
    # Display interactive HiPlot
    exp.display()
else:
    print("⚠️ HiPlot not available. Install with: pip install hiplot")
    print("\nAlternatively, use the parallel coordinates plot above.")

## 7. Statistical Analysis

In [ ]:
# Calculate summary statistics
stats = analyzer.calculate_summary_statistics()

print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

for metric, metric_stats in stats.items():
    print(f"\n📊 {metric}:")
    print(f"  Mean:   {metric_stats['mean']:.6f}")
    print(f"  Std:    {metric_stats['std']:.6f}")
    print(f"  Min:    {metric_stats['min']:.6f}")
    print(f"  Max:    {metric_stats['max']:.6f}")
    print(f"  Median: {metric_stats['median']:.6f}")
    print(f"  Count:  {metric_stats['count']}")

In [ ]:
# Correlation heatmap
correlations = analyzer.calculate_correlations(CONFIG["primary_metric"])

if correlations:
    # Convert to DataFrame for visualization
    corr_df = pd.DataFrame(correlations)
    
    # Create heatmap
    fig_corr = go.Figure(data=go.Heatmap(
        z=corr_df.values,
        x=[c.replace("config_", "").replace("metric_", "") for c in corr_df.columns],
        y=[c.replace("config_", "").replace("metric_", "") for c in corr_df.index],
        colorscale="RdBu",
        zmid=0,
    ))
    
    fig_corr.update_layout(
        title="Parameter-Metric Correlations",
        width=800,
        height=600,
    )
    
    fig_corr.show()
else:
    print("No numeric columns for correlation analysis.")

## 8. Generate Report

In [ ]:
# Generate comprehensive report
report = analyzer.generate_report(
    metric=CONFIG["primary_metric"],
    mode=CONFIG["metric_mode"],
)

# Display as markdown
display(Markdown(report.to_markdown()))

In [ ]:
# Save report and figures
output_dir = Path(CONFIG["output_dir"])

# Save report as JSON
report.to_json(output_dir / "analysis_report.json")
print(f"✅ Report saved to {output_dir / 'analysis_report.json'}")

# Save report as Markdown
with open(output_dir / "analysis_report.md", "w") as f:
    f.write(report.to_markdown())
print(f"✅ Markdown report saved to {output_dir / 'analysis_report.md'}")

# Save figures
analyzer.save_figure(fig_history, str(output_dir / "optimization_history.html"))
print(f"✅ Optimization history saved")

analyzer.save_figure(fig_importance, str(output_dir / "parameter_importance.html"))
print(f"✅ Parameter importance saved")

# Export data to CSV
analyzer.export_to_csv(str(output_dir / "tuning_results.csv"))
print(f"✅ Results exported to CSV")

## 9. Run New Tuning Experiment (Optional)

Launch a new hyperparameter tuning experiment from this notebook.

In [ ]:
# Configure new tuning experiment
tuning_config = {
    "model_name": CONFIG["model_name"],
    "num_samples": 20,  # Number of trials
    "scheduler": "asha",  # "asha" or "pbt"
    "search_alg": "optuna",  # "optuna" or "random"
    "train_data_path": "../data/train.jsonl",
    "eval_data_path": "../data/eval.jsonl",
}

# Display configuration
print("Tuning Configuration:")
for key, value in tuning_config.items():
    print(f"  {key}: {value}")

In [ ]:
# Interactive tuning configuration
num_samples_slider = widgets.IntSlider(
    value=tuning_config["num_samples"],
    min=5,
    max=100,
    step=5,
    description="Num Samples:",
    style={"description_width": "initial"},
)

scheduler_dropdown = widgets.Dropdown(
    options=["asha", "pbt", "fifo"],
    value=tuning_config["scheduler"],
    description="Scheduler:",
    style={"description_width": "initial"},
)

search_alg_dropdown = widgets.Dropdown(
    options=["optuna", "random"],
    value=tuning_config["search_alg"],
    description="Search Alg:",
    style={"description_width": "initial"},
)

def update_tuning_config(change):
    tuning_config["num_samples"] = num_samples_slider.value
    tuning_config["scheduler"] = scheduler_dropdown.value
    tuning_config["search_alg"] = search_alg_dropdown.value

num_samples_slider.observe(update_tuning_config, names="value")
scheduler_dropdown.observe(update_tuning_config, names="value")
search_alg_dropdown.observe(update_tuning_config, names="value")

display(widgets.VBox([
    widgets.HTML("<h4>Tuning Configuration</h4>"),
    num_samples_slider,
    widgets.HBox([scheduler_dropdown, search_alg_dropdown]),
]))

In [ ]:
# Preview search space
search_space = SearchSpaceConfig.for_model(tuning_config["model_name"])

print("\n" + "="*60)
print(f"SEARCH SPACE FOR {tuning_config['model_name'].upper()}")
print("="*60)

for name, param in search_space.parameters.items():
    if param.type == "categorical":
        print(f"  {name}: choices = {param.choices}")
    elif param.type == "log_uniform":
        print(f"  {name}: log_uniform({param.lower:.2e}, {param.upper:.2e})")
    else:
        print(f"  {name}: {param.type}({param.lower}, {param.upper})")

In [ ]:
# Run tuning (uncomment to execute)
# ⚠️ This will start actual tuning which may take significant time and resources

# run_tuning_button = widgets.Button(
#     description="🚀 Start Tuning",
#     button_style="danger",
#     layout=widgets.Layout(width="200px", height="50px"),
# )
# 
# tuning_output = widgets.Output()
# 
# def run_tuning(button):
#     with tuning_output:
#         tuning_output.clear_output()
#         print("Starting hyperparameter tuning...")
#         print(f"Model: {tuning_config['model_name']}")
#         print(f"Trials: {tuning_config['num_samples']}")
#         print(f"Scheduler: {tuning_config['scheduler']}")
#         print(f"Search Algorithm: {tuning_config['search_alg']}")
#         print("\n" + "-"*40)
#         
#         # Create tuner
#         tune_config = TuneConfig.for_model(tuning_config["model_name"])
#         tune_config.num_samples = tuning_config["num_samples"]
#         tune_config.scheduler_type = tuning_config["scheduler"]
#         tune_config.search_alg_type = tuning_config["search_alg"]
#         
#         tuner = HyperparameterTuner(
#             model_name=tuning_config["model_name"],
#             search_space=search_space,
#             tune_config=tune_config,
#         )
#         
#         # Run tuning
#         result = tuner.run(
#             train_data_path=tuning_config["train_data_path"],
#             eval_data_path=tuning_config["eval_data_path"],
#         )
#         
#         print("\n✅ Tuning complete!")
#         print(f"Best config: {result.best_config}")
#         print(f"Best metrics: {result.best_metrics}")
# 
# run_tuning_button.on_click(run_tuning)
# display(widgets.VBox([run_tuning_button, tuning_output]))

print("💡 To run tuning, uncomment the code above and click the button.")
print("⚠️ Tuning requires GPU resources and may take several hours.")

## 10. Compare Multiple Experiments

In [ ]:
# Load and compare multiple experiments
experiment_paths = {
    # Add paths to experiments you want to compare
    # "experiment_1": "../ray_results/experiment_1",
    # "experiment_2": "../ray_results/experiment_2",
}

if experiment_paths:
    experiments = {}
    for name, path in experiment_paths.items():
        if Path(path).exists():
            exp_analyzer = HPAnalyzer()
            exp_analyzer.load_results_from_path(path)
            experiments[name] = exp_analyzer
            print(f"✅ Loaded {name}: {exp_analyzer.num_trials} trials")
    
    if len(experiments) > 1:
        from medai_compass.optimization.analysis import create_comparison_plot
        
        fig_compare = create_comparison_plot(
            experiments=experiments,
            metric=CONFIG["primary_metric"],
            mode=CONFIG["metric_mode"],
        )
        fig_compare.show()
    else:
        print("Need at least 2 experiments to compare.")
else:
    print("💡 Add experiment paths to compare multiple tuning runs.")
    print("   Edit the experiment_paths dictionary in the cell above.")

## 11. Conclusion & Next Steps

### Summary

This notebook provided comprehensive hyperparameter analysis including:
- Best configuration identification
- Parameter importance analysis
- Interactive visualizations
- Statistical summaries
- Report generation

### Next Steps

1. **Train with best config**: Use the best configuration to train your final model
2. **Fine-tune search space**: Based on importance analysis, narrow the search space
3. **Try PBT**: If using ASHA, consider Population-Based Training for adaptive tuning
4. **Scale up**: Increase num_samples for more thorough exploration

In [ ]:
# Print final recommendations
print("\n" + "="*60)
print("📋 RECOMMENDATIONS")
print("="*60)

for i, rec in enumerate(report.recommendations, 1):
    print(f"\n{i}. {rec}")

print("\n" + "="*60)
print("Analysis complete! 🎉")
print("="*60)